# **FOREWORD**

This is a crude baseline for the [Playground Season-6 Episode-5]() competition. This is a binary classifier with roc-auc metric. 

We are predicting the probability of a pit-stop in the next lap here. This metric needs to be maximized.

I start with the kernel [here](https://www.kaggle.com/code/emanuellcs/predicting-f1-pit-stops-xgboost-baseline). I make the below changes to this work- 
- Change the CV scheme to StratifiedGroupKFold
- Add target encoder
- Use additional features on race and driver aggregates 

# **IMPORTS**

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata

from itertools import *
from gc import collect
from tqdm.auto import tqdm

from sklearn.preprocessing import *
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import clone
from sklearn.model_selection import *
from sklearn.metrics import *

import xgboost as xgb
from xgboost import XGBClassifier as XGBC, DMatrix

import warnings
warnings.simplefilter('ignore')

# **CONFIGURATION**

In [ ]:

# CONFIGURATION & PATHS
COMP_PATH = '/kaggle/input/competitions/playground-series-s6e5/'
ORIG_PATH = '/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/'
SUB_PATH  = '/kaggle/input/datasets/ravi20076/playgrounds6e5publictestpreds/test_preds.csv'

TARGET       = 'PitNextLap'
RANDOM_STATE = 42
TEST_REQ     = False
USE_ORIG     = False
BUILD_MODEL  = False

if TEST_REQ:
    DEVICE       = 'cpu' 
    NEST         = 10
    N_SPLITS     = 2
else:
    DEVICE       = 'cuda' 
    NEST         = 5000 
    N_SPLITS     = 5

# **PREPROCESSING**
We will load the competition data and the original F1 dataset. Since the competition explicitly states `Normalized_TyreLife` makes the prediction trivial and was removed from the test set, we must ensure we drop it from the original dataset before merging.

In [ ]:
# LOAD DATA

print("Loading data")

train_comp = pd.read_csv(COMP_PATH + 'train.csv')
test       = pd.read_csv(COMP_PATH + 'test.csv')
subm       = pd.read_csv(COMP_PATH + 'sample_submission.csv')

# Load original dataset
if USE_ORIG:
    train_orig = pd.read_csv(ORIG_PATH + 'f1_strategy_dataset_v4.csv') 
    print("Original dataset loaded successfully.")
    
    # Drop the trivial feature if it exists in the original data
    if 'Normalized_TyreLife' in train_orig.columns:
        train_orig = train_orig.drop(columns=['Normalized_TyreLife'])
        
    # Align columns: only keep features that exist in the competition training set
    common_cols = [col for col in train_comp.columns if col in train_orig.columns]
    train_orig = train_orig[common_cols]
    
    # Merge datasets and drop duplicates
    train = pd.concat([train_comp, train_orig], axis=0).reset_index(drop=True)
    train = train.drop_duplicates()
    
else:
    print("Proceeding with competition data only")
    train = train_comp.copy()

train = train.rename(columns = {'LapTime (s)': 'LapTimes'})
test  = test.rename(columns = {'LapTime (s)': 'LapTimes'})

print(f"Final Train shape: {train.shape}; Test shape: {test.shape}")

# **FEATURE PREPARATION**


I add extra features to his pipeline to test the model on them!!

In [ ]:
features = [feat for feat in test.columns if feat != 'id']
cat_cols = test.select_dtypes(include=['object', 'category']).columns.tolist()

X        = train[features].copy()
y        = train[TARGET].copy()
X_test   = test[features].copy()

num_cols = list(set(features).difference(set(cat_cols)))
combined = pd.concat([X , X_test], axis=0, ignore_index = True)

# Category twins of numeric features
for col in num_cols:
    combined[f"C{col}"] = combined[col].astype("string").fillna("missing")
    enc = LabelEncoder()
    combined[f"C{col}"] = enc.fit_transform(combined[f"C{col}"].values)
    combined[f"C{col}"] = combined[f"C{col}"].astype("string")
    cat_cols.append(f"C{col}")

# Frequency Encoding for categorical features
for col in cat_cols:
    freq_map               = combined[col].value_counts(normalize=True).to_dict()
    combined[f'GCE_{col}'] = combined[col].map(freq_map)

# Global feature-encoder for numeric features
agg_cols     = list(set(num_cols).difference(set(["Driver", "Compound", "Race", "Year"])))
groupby_keys = ["Driver", "Compound", "Race", "Year"]

for agg_func, prefix in [("mean", "GFEMean_"), ("max", "GFEMax_"), ("min", "GFEMin_")]:
    df = (
        combined.groupby(groupby_keys)[agg_cols]
        .agg(agg_func)
        .add_prefix(prefix)
        .reset_index()
    )
    combined = combined.merge(df, how="left", on=groupby_keys)

# Bigrams for cat-features
combs = list(combinations(cat_cols, 2))
for col1, col2 in combs:
    combined[f"{col1}-{col2}"] = combined[col1].fillna("missing") + "-" + combined[col2].fillna("missing")
    cat_cols.append(f"{col1}-{col2}")

cat_cols           = combined.select_dtypes(exclude = np.number).columns.tolist()
combined[cat_cols] = combined[cat_cols].astype("string").fillna("missing")
features           = combined.columns.tolist()

with np.printoptions(linewidth = 150):
    print(np.array(features))
    print(f"\n---> Final feature shape = {combined.shape}\n")

Xtrain = combined.iloc[0: -1*len(X_test)]
Xtest  = combined.iloc[-1*len(X_test):]
ytrain = train[TARGET].copy()

del X, y, X_test, combined
print(f"\n---> Final Shape = {Xtrain.shape} {ytrain.shape} {Xtest.shape}\n")
collect();

# **MODEL TRAINING**


In [ ]:
# MODEL TRAINING & CV

if BUILD_MODEL :
    
    cv  = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    grp = Xtrain["Driver"] + "-" + Xtrain["Race"] 
    
    oof_preds  = np.zeros(len(Xtrain))
    test_preds = 0
    ftreimp    = 0
    
    ct = ColumnTransformer(
        [("TE", TargetEncoder(random_state = 42), cat_cols)],
        remainder = "passthrough",
        verbose_feature_names_out = False
    )
    
    mdl = XGBC(**{
            'n_estimators'       : NEST,
            'learning_rate'      : 0.007,
            'max_depth'          : 7,
            'subsample'          : 0.8,
            'colsample_bytree'   : 0.8,
            'objective'          : 'binary:logistic',
            'eval_metric'        : 'auc',
            'tree_method'        : 'hist',
            'random_state'       : RANDOM_STATE,
            'device'             : DEVICE,
            'n_jobs'             : -1,
            'enable_categorical' : True,
            'max_bin'            : 1024,
        }
    )
    
    pipe = Pipeline(
        steps = [
            ("PP", ct),
            ("M", mdl)
        ]
    )
    
    print(f"\n---> Starting Offline Training\n")
    
    for fold, (train_idx, valid_idx) in tqdm(enumerate(cv.split(Xtrain, ytrain, grp))):
    
        X_train_fold, y_train_fold = Xtrain.iloc[train_idx], ytrain.iloc[train_idx]
        X_valid_fold, y_valid_fold = Xtrain.iloc[valid_idx], ytrain.iloc[valid_idx]
        
        model = clone(pipe)
        model.fit(X_train_fold, y_train_fold)
        ftreimp += model["M"].feature_importances_ / N_SPLITS
        
        valid_fold_preds = model.predict_proba(X_valid_fold)[:, 1]
        oof_preds[valid_idx] = valid_fold_preds
    
        test_preds += model.predict_proba(Xtest)[:, 1] / N_SPLITS
        
        fold_auc = roc_auc_score(y_valid_fold, valid_fold_preds)
        print(f"---> Score = {fold_auc:.8f} | Fold {fold}")
    
    # Calculate overall Out-Of-Fold AUC
    oof_auc = roc_auc_score(ytrain, oof_preds)
    print(f"\n---> Overall Score = {oof_auc:.8f}\n\n")
    
    try:
        FtreImp = pd.DataFrame(
            np.stack([model["M"].feature_names_in_, ftreimp], axis=1),
            columns = ["Feature", "Importance"]
        )
        
        FtreImp = FtreImp.set_index("Feature")
        
        display(
            FtreImp
            .sort_values("Importance", ascending = False)
            .head(25)
            .style
            .set_caption("Top Important Features")
        )
    except:
        pass

else:
    print(f"---> Loading model submission from past-version")
    test_preds = pd.read_csv(SUB_PATH)[TARGET].values.flatten()


# **SUBMISSION**


In [ ]:

subm[TARGET] = test_preds

# Blending with public work-
sub = pd.read_csv(
    f"/kaggle/input/datasets/ravi20076/playgrounds6e5publictestpreds/submission.csv"
)[TARGET].values

subm[TARGET] = (
    -0.05   * rankdata(subm[TARGET].values, method = "dense") / (len(subm) + 1)
    + 1.05 * rankdata(sub, method = "dense") / (len(subm) + 1)
)
subm.to_csv('submission.csv', index=False)
print()
display(subm.head())